# Setup

In [ ]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

import glob

from skimage import io, util, filters, morphology, measure, color

from scipy.ndimage import binary_fill_holes

import os
import json

In [ ]:
input  = '/content/drive/MyDrive/Vision/Prepared/1_coins/'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Loading

In [ ]:
df = pd.read_csv(input + 'labels.csv', sep=',')


In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.info

In [ ]:
files = glob.glob(input + '*.png')

In [ ]:
for file in files[:1]:

  image = io.imread(file)
  image = util.img_as_ubyte(image)
  edges = filters.sobel(image)

  low = 0.09
  high = 0.20

  # aplica o hysteresis threshold
  hyst = filters.apply_hysteresis_threshold(edges, low, high)

  binary = hyst > 0
  binary = morphology.remove_small_objects(binary, min_size=300)
  binary = binary_fill_holes(binary)
  binary = morphology.binary_closing(binary, morphology.disk(5))

  label_img = measure.label(binary)
  num_coins = label_img.max()
  print(f"Imagem: {file} → Moedas detectadas: {num_coins}")

  # cria figura (1 por imagem)
  fig, ax = plt.subplots(1, 3, figsize=(12, 4))
  ax[0].imshow(image, cmap='gray')
  ax[0].set_title('Original')
  ax[1].imshow(hyst, cmap='magma')
  ax[1].set_title('Hysteresis threshold')
  ax[2].imshow(color.label2rgb(label_img, image=image))
  ax[2].set_title(f'Moedas detectadas: {num_coins}')

  for a in ax:
      a.axis('off')

  plt.tight_layout()
  plt.show()
  plt.close(fig)